In [ ]:
"""
import the vowpal wabbit library.
"""
pip install vowpalwabbit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 12.3 MB/s eta 0:00:00


In [ ]:
"""
import the other required libraries.
"""
import pandas as pd
import vowpalwabbit
import numpy as np
import json

model = 'word2vec'

In [ ]:
"""
load the genre_mapping dictioanry from the file directory. The genre_mapping dictionary is created once and used for all the three models. It contains the numerical representation of genres.
"""
genres_mappings = dict()
txt_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/input/genre_mapping.txt'
with open(txt_file_path, 'r') as txt_file:
    genres_mappings = json.loads(txt_file.read())
print(genres_mappings)

unique_genres = list(genres_mappings.values())
print(unique_genres)

{'Crime': 1, 'Western': 2, 'Fantasy': 3, 'Science Fiction': 4, 'Comedy': 5, 'Family': 6, 'Mystery': 7, 'Drama': 8, 'Animation': 9, 'Thriller': 10, 'Documentary': 11, 'Horror': 12, 'Romance': 13, 'TV Movie': 14, 'History': 15, 'War': 16, 'Adventure': 17, 'Music': 18, 'Action': 19}
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
"""
load the dataset to be used for the experiment. This dataset was created by the data prepare step. This will act as the main dataset for this experiment also referred as the duplicate_instances_dataset in the report and will be refered as action_context_map for the experiment code.
"""
action_context_map = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_action_context_map.csv')

In [ ]:
"""
convert the textual genres to the numerical genres columns.
"""
from ast import literal_eval
action_context_map['genres_indices'] = action_context_map['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of action_context_map.
"""
action_context_map.head(100)

,genres_index,keywords_embedding_string,genres_indices,id,original_title_embedding_string,overview_embedding_string,release_date
0,1,2.85971182e-05 1.22066260e-04 7.85888187e-05...,"[11, 8]",1114,5.49113553e-04 4.32684101e-05 -3.88299057e-04...,-1.54813490e-04 -3.44812201e-04 -1.07973814e-0...,6/29/2003
1,1,-2.33949628e-04 -2.51401099e-04 -1.62103843e-0...,[8],1115,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,-1.23861959e-04 -2.31350510e-04 6.22942316e-0...,9/15/2005
2,1,1.55269503e-04 7.78855538e-05 3.00870626e-04...,"[8, 16]",1116,2.61010689e-04 5.95547026e-05 -3.53123207e-04...,-5.10586069e-05 -2.65968934e-04 1.42931036e-0...,6/23/2006
3,1,2.60780536e-04 2.01622956e-04 8.58218118e-05...,"[19, 8, 10]",1123,6.32884854e-04 1.68751998e-04 1.46541162e-04...,-1.95092307e-05 -5.33183862e-04 5.61742054e-0...,10/27/2006
4,1,-7.94404623e-05 6.00340863e-05 -1.00449332e-0...,"[8, 7, 4]",1124,1.15597527e-03 -9.27828252e-04 -1.09264103e-03...,3.73611219e-05 -5.75762242e-04 -1.30056069e-04...,10/19/2006
...,...,...,...,...,...,...,...
95,1,-3.45864159e-04 2.83954752e-04 -5.02702431e-0...,"[13, 8]",1413,7.69345846e-04 1.06385283e-04 -4.71321750e-04...,-2.56165105e-04 -8.30602512e-05 -2.78349180e-0...,10/1/1993
96,1,2.01106941e-05 -1.81871044e-04 -4.20956858e-05...,"[8, 7, 13]",1414,1.50432068e-04 -4.00678953e-04 -3.56293662e-04...,-3.37247948e-05 -4.01045225e-04 -7.34196728e-0...,9/4/1998
97,1,1.87221485e-05 -4.29044798e-04 -7.68041573e-05...,"[5, 8, 1]",1415,-7.89361191e-04 -1.52942986e-04 -4.49090148e-0...,2.20442816e-04 -2.66293966e-04 3.99194796e-05...,9/5/2003
98,1,-2.03096439e-04 -6.73356350e-04 -1.30666649e-0...,"[8, 16]",1416,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,-5.28203855e-05 -3.58069141e-04 1.09576853e-0...,11/22/2002


In [ ]:
"""
print and see the size of action_context_map
"""
action_context_map.shape[0]

190000

In [ ]:
"""
a variable to ensure test dataset is selected at random from the entire dataset.
"""
random_test_data_flag = True

In [ ]:
"""
the test dataset will always be the random test dataset. But incase if the experiment needs to be performed for a test data for which the model has prior knowledge, the set the flag to False in the above line.
"""
if random_test_data_flag:
  test_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_RANDOM_TEST_file.csv')
else:
  test_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TEST_file.csv')

test_movies_df['genres_indices'] = test_movies_df['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of test dataset.
"""
test_movies_df.head(10)

,keywords_embedding_string,genres_indices,id,original_title_embedding_string,release_date,overview_embedding_string
0,-2.71542260e-04 3.84506508e-04 3.85096297e-0...,[11],1223382,4.87703044e-04 8.64601054e-04 5.29336568e-04...,11/3/2023,1.51260756e-05 -7.63295684e-04 9.82480851e-05...
1,-3.92511167e-04 -7.41362674e-05 9.50358008e-0...,[5],583413,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,9/29/1984,-2.12200888e-04 -4.29416599e-04 -1.84895864e-0...
2,4.61565032e-05 -6.42323575e-04 4.97625035e-04...,[11],342852,2.54323240e-04 -1.15310203e-03 -2.16587679e-04...,1/1/1968,-1.70094092e-04 -6.88310654e-04 9.93461945e-0...
3,1.75603083e-04 7.79393449e-05 -1.87525176e-04...,"[8, 5, 13]",10401,-2.41370522e-04 -1.07065811e-04 -5.85233793e-0...,3/31/1999,3.06024413e-05 -4.17455885e-04 1.16513846e-04...
4,-4.59445037e-05 -3.88514600e-04 -3.51291616e-0...,[11],802160,-5.11612277e-04 -3.08351271e-04 -6.11693598e-0...,10/20/2000,-9.65296204e-05 -3.60660575e-04 -2.04455773e-0...
5,-2.39511995e-04 -5.43001079e-06 4.42468008e-0...,"[4, 12]",1016490,-7.17788353e-04 1.92671607e-04 5.36523468e-0...,NaN,2.61569712e-05 -6.30986295e-04 -2.89209594e-04...
6,4.16768395e-04 1.09134999e-05 1.05295051e-03...,[11],331916,-1.09750268e-04 -2.41750779e-04 7.02748948e-0...,9/29/2012,4.65087978e-05 -1.13877933e-04 -9.35278003e-05...
7,1.29867316e-04 2.59571389e-04 2.62751157e-04...,[11],45558,2.56642554e-04 -8.09424964e-05 1.85882891e-05...,10/14/1992,-9.39023885e-05 -3.26972746e-04 3.98901429e-0...
8,-1.41537530e-04 7.10706590e-05 3.13618075e-0...,"[12, 7]",37726,0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0...,10/6/1998,-5.63221438e-05 -2.93313089e-04 2.69579643e-0...
9,6.92070462e-04 -8.88146926e-04 1.49818414e-04...,[8],1142678,-1.05888954e-04 1.24803060e-04 1.44827791e-0...,3/2/2019,2.58251748e-05 -4.46360791e-04 2.97881015e-05...


In [ ]:
"""
print and see the size of action_context_map
"""
test_movies_df.shape[0]

38000

In [ ]:
"""
load the train dataset to be used for the experiment. This dataset was created by the data prepare step.
"""
train_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TRAIN_file.csv')
train_movies_df['genres_indices'] = train_movies_df['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of train dataset.
"""
train_movies_df.head(10)

,genres_index,keywords_embedding_string,genres_indices,id,original_title_embedding_string,overview_embedding_string,release_date,rewards,cost
0,1,1.35878247e-04 -4.99582238e-05 -1.77093956e-04...,"[6, 9, 13, 3]",14128,1.53836590e-04 -1.06899672e-04 1.81367941e-04...,-9.86703890e-05 -1.89863364e-04 6.71631060e-0...,2/23/2002,0.0,1.0
1,5,-6.31776275e-05 -1.83687880e-04 3.69781745e-0...,"[6, 3, 9, 14, 18, 7, 17]",13351,3.17973492e-04 1.82243675e-04 -1.76852136e-05...,-9.16904082e-06 -3.58913763e-04 1.28509084e-0...,9/3/1994,0.0,1.0
2,18,-2.29682963e-04 -3.88696644e-04 5.73111720e-0...,"[8, 7, 10]",1126,4.90836101e-07 2.24936797e-04 5.28393648e-05...,-7.85529483e-05 -4.99353977e-04 1.05275627e-0...,10/7/2004,0.0,1.0
3,1,-3.77695425e-04 -2.85167247e-04 -1.22307916e-0...,"[5, 12, 3]",16846,-1.13783775e-04 2.85410642e-04 1.03768172e-0...,-9.24519845e-05 -4.11116023e-04 8.10430502e-0...,7/4/2008,0.0,1.0
4,3,1.01052562e-03 -3.01140273e-04 -9.29304806e-04...,"[19, 5, 1]",13534,1.08377732e-04 -1.11960318e-04 6.95276249e-04...,-1.03192133e-06 -3.72458919e-04 4.17866613e-0...,1/25/2007,0.0,1.0
5,4,-7.20823708e-04 -3.99504439e-04 7.15998394e-0...,"[19, 1, 10]",14398,5.06591692e-04 -3.61051643e-04 -1.85572004e-04...,-2.63709342e-04 -3.57329962e-04 6.02750733e-0...,10/8/1997,0.0,1.0
6,2,-4.54943831e-04 -5.69693570e-04 -3.32579657e-0...,"[12, 4, 7, 8]",9682,-1.13219826e-03 4.04885504e-04 3.99277400e-0...,1.61833799e-04 -5.30342513e-04 2.11473089e-04...,8/11/2006,0.0,1.0
7,19,4.95426284e-05 1.08286804e-04 2.99298845e-05...,"[5, 1]",11872,5.14564221e-04 -7.89543774e-05 -3.33500298e-04...,7.40684045e-05 -3.20937688e-04 2.64925420e-05...,10/19/1962,0.0,1.0
8,5,1.04513201e-04 -1.29440392e-04 6.53695824e-05...,"[10, 12]",18781,6.05803623e-04 -7.49074155e-04 -1.99332091e-04...,-1.78650283e-04 -3.88228276e-04 1.85887955e-0...,3/27/2009,0.0,1.0
9,14,1.80026895e-04 3.60942271e-04 -1.59692456e-04...,"[19, 5, 8]",15836,-4.71967418e-04 -4.93271626e-04 3.00685555e-0...,-9.85882507e-05 -7.76480374e-05 -2.92333571e-0...,3/9/2007,0.0,1.0


In [ ]:
"""
print and see the records of train dataset.
"""
train_movies_df.shape[0]

152000

In [ ]:
"""
print distribution of movie genres
"""
all_numbers = [num for sublist in train_movies_df['genres_indices'] for num in sublist]
counts_df = pd.DataFrame(all_numbers, columns=['Number']).groupby('Number').size().reset_index(name='Count')
print(counts_df)

    Number  Count
0        1  22191
1        2   3682
2        3  11736
3        4  13099
4        5  57950
5        6  13376
6        7  10038
7        8  69035
8        9   8436
9       10  29987
10      11   7064
11      12  17500
12      13  25783
13      14   4357
14      15   6450
15      16   5004
16      17  19352
17      18   6261
18      19  27299


In [ ]:
"""
load the vowpal wabbit formatted train dataset
"""
vw_train_data = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_vw_train_data.txt')

In [ ]:
"""
load the vowpal wabbit model and train it on train data
"""
vw_model = vowpalwabbit.Workspace(f"--cb_explore {len(unique_genres)} --quiet --epsilon 0.1")
for example in vw_train_data:
    vw_model.learn(example)

In [ ]:
"""
save the trained model
"""
vw_model.save(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_cb_EP-0.1.model')

In [ ]:
"""
this function assigns the cost to the predicted action
"""
def test_assign_cost(predicted_action, movie_id):
    action_context_map = test_movies_df
    search_col = 'id'
    search_val = int(movie_id)
    target_column = 'genres_indices'
    res = action_context_map[(action_context_map[search_col] == search_val)]
# uses the same mathematical formula as stated in the report equation 7
    ind = -1
    index_list = res[target_column].iloc[0]
    if predicted_action in index_list:
      ind = index_list.index(predicted_action)
      if ind == 0:
          return 0
      elif ind >= 1 :
        return ind/len(index_list)
    else:
      return 1

In [ ]:
"""
this function is the probability mass function or pmf.  It takes a list pmf as input, which represents the probabilities of each outcome.
"""
import random
def sample_custom_pmf(pmf):
    total = sum(pmf)
    scale = 1 / total
    pmf = [x * scale for x in pmf]
    draw = random.random()
    sum_prob = 0.0
    for index, prob in enumerate(pmf):
        sum_prob += prob
        if sum_prob > draw:
            return index, prob

In [ ]:
"""
this function is to define the prediction in the model. It passes the action, cost and context in the vowpal wabbit format and triggers the model to make predictions
"""
def predict_genre(row):
    formatted_row = row["keywords_embedding_string"]
    formatted_id = row["id"]
    # - for all attr
    formatted_original_title_embedding_string = row["original_title_embedding_string"]
    formatted_release_date = row["release_date"]
    formatted_overview_embedding_string = row["overview_embedding_string"]
    # - for only 2 attr
    # context_keywords = f'| feature1:{formatted_id} feature2:{formatted_row}'
    # - for all attr
    context_keywords = f'| feature1:{formatted_id} feature2:{formatted_row} feature3:{formatted_original_title_embedding_string} feature4:{formatted_release_date} feature5:{formatted_overview_embedding_string}'
    action_probabilities = vw_model.predict(context_keywords)
    chosen_index, prob = sample_custom_pmf(action_probabilities)
    return (chosen_index + 1, prob, action_probabilities)

In [ ]:
"""
this function is used to run the prediction iterations.
It stores the results of each iterations in specified files like rewards, regret, accuracy, conversion rate
"""
num_iterations = 300
res = []

totalObtainedRewardsSum = 0
totalObtainedCostSum = 0
iterationRewardsCounter = 0
iterationCostCounter = 0
iterationRewardsList = []
iterationCostList = []
actionCounts = {}
successCounts = {}
accurateCounter = 0

for iteration in range(num_iterations):
  shuffled_indices = np.random.permutation(len(test_movies_df))
  for index in shuffled_indices:
    row = test_movies_df.iloc[index]
    resl = predict_genre(row)
    predicted_genre, prob = resl[0], resl[1]
    keywords_string = row['keywords_embedding_string']
    movie_id = row['id']
    original_title_embedding_string = row["original_title_embedding_string"]
    release_date = row["release_date"]
    overview_embedding_string = row["overview_embedding_string"]
    actual_cost = test_assign_cost(predicted_genre, movie_id)
    totalObtainedRewardsSum += (1-actual_cost)
    iterationRewardsCounter += (1-actual_cost)
    totalObtainedCostSum += actual_cost
    iterationCostCounter += actual_cost

    if predicted_genre not in actionCounts:
        actionCounts[predicted_genre] = 0
    actionCounts[predicted_genre] += 1

    if predicted_genre in row["genres_indices"]:
        accurateCounter += 1
        if predicted_genre not in successCounts:
            successCounts[predicted_genre] = 0
        successCounts[predicted_genre] += 1

    vw_model.learn(f'{predicted_genre}:{actual_cost}:{prob} | feature1:{movie_id} feature2:{keywords_string} feature3:{original_title_embedding_string} feature4:{release_date} feature5:{overview_embedding_string}')

  iterationRewardsList.append(iterationRewardsCounter)
  iterationCostList.append(iterationCostCounter)
  iterationRewardsCounter = 0
  iterationCostCounter = 0

conversion_rates = {}
for action, success_count in successCounts.items():
    total_count = actionCounts[action]
    conversion_rate = success_count / total_count
    conversion_rates[action] = conversion_rate

modelAccuracy = accurateCounter / (len(test_movies_df) * num_iterations)

In [ ]:
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_res_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in res:
        file.write("%s\n" % item)

In [ ]:
"""
print the cumulative rewards over 300 iterations
"""
print(totalObtainedRewardsSum)

5691375.135321818


In [ ]:
"""
print the cumulative regret over 300 iterations
"""
print(totalObtainedCostSum)

5708624.864676793


In [ ]:
"""
print the cumulative rewards after end of each iteration
"""
print(iterationRewardsList)

[8685.092857142869, 11698.00595238093, 11906.601190476169, 13023.667857142804, 12908.618650793609, 14322.763095238015, 13642.080952380897, 14943.744047618953, 14875.553571428463, 15057.677380952282, 15220.201190476084, 15360.969047618944, 15341.660714285626, 15730.745238095147, 15380.609920634824, 15608.459523809417, 15689.60833333322, 15818.307142857038, 15982.44642857131, 16260.015476190372, 16639.92380952368, 16140.71071428557, 16487.19523809511, 16320.552380952238, 16779.74404761892, 16161.082142857009, 16103.230952380822, 16778.41190476178, 16382.202380952223, 17056.39880952371, 16509.227380952274, 16445.855158730046, 16823.53015873002, 16944.190873015756, 17007.879761904667, 17240.965476190395, 16986.651190476074, 17606.59880952376, 17032.024999999867, 17207.03809523798, 17404.52261904751, 17411.959523809404, 17243.4988095237, 16940.86547619039, 17584.999999999924, 17422.771428571337, 17320.09285714275, 17226.487301587178, 17438.01666666656, 17938.942857142774, 17828.64285714278,

In [ ]:
"""
print the cumulative regret after end of each iteration
"""
print(iterationCostList)

[29314.90714285711, 26301.994047618988, 26093.39880952378, 24976.33214285707, 25091.381349206302, 23677.236904761838, 24357.91904761899, 23056.255952380903, 23124.44642857132, 22942.322619047558, 22779.798809523778, 22639.030952380956, 22658.339285714253, 22269.254761904733, 22619.39007936507, 22391.540476190476, 22310.39166666662, 22181.692857142814, 22017.553571428474, 21739.984523809453, 21360.07619047616, 21859.289285714207, 21512.804761904703, 21679.447619047565, 21220.255952380976, 21838.91785714282, 21896.76904761901, 21221.588095238, 21617.797619047604, 20943.601190476198, 21490.772619047573, 21554.144841269772, 21176.469841269798, 21055.809126984095, 20992.12023809524, 20759.034523809492, 21013.34880952378, 20393.401190476186, 20967.97500000001, 20792.96190476188, 20595.477380952434, 20588.040476190454, 20756.501190476134, 21059.134523809502, 20415.00000000002, 20577.228571428564, 20679.907142857133, 20773.512698412695, 20561.98333333334, 20061.057142857135, 20171.357142857134

In [ ]:
"""
save the cumulative rewards data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_reward_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in iterationRewardsList:
        file.write("%s\n" % item)

In [ ]:
"""
save the cumulative regret data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_cost_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in iterationCostList:
        file.write("%s\n" % item)

In [ ]:
"""
save the conversion data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_conversion_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
  for action, conversion_rate in conversion_rates.items():
    file.write("%s %s\n" % (action, conversion_rate))

In [ ]:
"""
save the accuracy data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_accuracy_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    file.write("%s \n" % modelAccuracy)

In [ ]:
# End